# 02 · EDA, vintage and censoring diagnostics
**Questions this notebook answers**
1. Is the validation window *observable*? (resolved share and maturity by vintage)
2. Does the default rate move by vintage for reasons that are economic vs. **censoring artefacts**?
3. Does missingness or population mix drift between train and validation?
4. Are there outliers that justify the winsorising rule?

### Why censoring dominates the design
A snapshot loan is "resolved" only once it is paid off or charged off. For vintages that are not yet fully matured, the resolved subset over-represents loans that defaulted **early** (and prepaid early), so the observed default rate is biased. Mixing those into a headline validation would distort calibration by decile - and therefore provisioning. Hence: clean validation = 2015 only; 2016 reported separately and labelled; 2017-18 for PSI only.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.pipeline import prepare
from src.reporting import vintage_table
data = prepare(cfg)
vt = vintage_table(data.df); vt

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(vt.vintage_year, vt.default_rate_resolved*100, color="#4c78a8"); ax.set_ylabel("Default rate, resolved loans (%)")
ax2 = ax.twinx(); ax2.plot(vt.vintage_year, vt.resolved_share*100, "o-", color="#e45756"); ax2.set_ylabel("Resolved share (%)")
for y in (2015.5,): ax.axvline(y, color="k", ls=":")
ax.set_title("Vintage default rate vs. resolved share (36m loans)  |  dotted = end of fully-matured window"); plt.tight_layout(); plt.show()

**Clean-validation maturity audit (issue month).** Snapshot = 2018-12-31; `fully_matured` is explicit date arithmetic (issue + 36 months <= snapshot). A month enters the clean sample only if fully matured **and** >= 95% of its loans are resolved; failing months (typically late-year ones at the window edge, e.g. Dec-2015) are demoted to the flagged sample and reported. Read this table before reading any 2015 result.

In [ ]:
data.valid_audit

**Censoring diagnostic by issue quarter.** If the resolved default rate jumps as the resolved share falls, that is the bias, not credit quality.

In [ ]:
q = data.df[data.df.vintage_year >= 2013].assign(q=lambda d: pd.to_datetime(d.issue_date).dt.to_period("Q").astype(str))
cq = q.groupby("q").agg(loans=("target","size"), resolved_share=("is_resolved","mean"), default_rate_resolved=("target","mean"),
                        matured=("fully_matured","mean")).reset_index()
cq

**Missingness by vintage** (train-vs-validation comparability).

In [ ]:
from src.data.cleaning import missingness_report
missingness_report(data.df, cfg["features"]["numeric"] + ["emp_length_raw"]).loc[2010:]

**Population drift:** FICO and DTI medians, loan size, by vintage (all statuses; used for PSI later).

In [ ]:
data.df.groupby("vintage_year")[["fico_mid","dti","annual_inc","loan_amnt","revol_util"]].median().loc[2010:]

**Outlier review.** Quantiles before vs. after train-fitted winsorising (rule R4). The point is that tail values (e.g. `annual_inc` in the millions, `dti` sentinels) cannot dominate the quantile bin edges.

In [ ]:
from src.data.cleaning import basic_clean
from src.data.loader import load_model_frame
before = basic_clean(load_model_frame(cfg)); tr_b = before[before.vintage_year.between(*cfg["split"]["train_years"]) & before.is_resolved.astype(bool)]
rows=[]
for c in ["annual_inc","dti","revol_util","loan_to_income"]:
    rows.append({"feature":c, "p0.5_raw":tr_b[c].quantile(.005), "p99.5_raw":tr_b[c].quantile(.995), "max_raw":tr_b[c].max(),
                 "max_after":data.train[c].max()})
pd.DataFrame(rows)

**Univariate risk ordering on train** (sanity check: does default rate move the way credit logic says?)

In [ ]:
tr = data.train.assign(fico_band=pd.cut(data.train.fico_mid,[0,660,680,700,720,740,900]), dti_band=pd.qcut(data.train.dti,5,duplicates="drop"))
for c in ["fico_band","dti_band","emp_length","home_ownership","purpose"]:
    print(tr.groupby(c, observed=True).target.agg(["size","mean"]).rename(columns={"mean":"default_rate"}).round(4), "\n")